# Bin Packing con Next Fit e First Fit
In questo esercizio risolviamo una istanza di Bin Packing Problem (BPP) con due euristiche: Next Fit e First Fit.

Vogliamo assegnare tutti gli item a un insieme di bin di capacita $C$, minimizzando il numero di bin usati.

## Formato file dati
Il file di input locale contiene:

1. numero di item $n$
2. capacita del bin $C$
3. i pesi $w_1, w_2, ..., w_n$ (uno per riga)

Esempio:

```
10
10
2
5
4
7
1
3
8
6
2
5
```

Tutti i pesi devono soddisfare $w_i \le C$.

In [1]:
class Item:
    def __init__(self):
        self.index = -1
        self.weight = -1
        self.bin = -1


def read_instance(path):
    with open(path, "r") as file_instance:
        list_of_string = [line.strip() for line in file_instance if line.strip() != ""]

    n_item = int(list_of_string[0])
    capacity = int(list_of_string[1])
    weights = [int(w) for w in list_of_string[2:]]

    if len(weights) != n_item:
        print("Attenzione: n diverso dal numero di pesi letti")

    list_of_item = []
    for i, w in enumerate(weights):
        new_item = Item()
        new_item.index = i
        new_item.weight = w
        list_of_item.append(new_item)

    return n_item, capacity, list_of_item


def clone_items(items):
    copied = []
    for old in items:
        new_item = Item()
        new_item.index = old.index
        new_item.weight = old.weight
        new_item.bin = -1
        copied.append(new_item)
    return copied


instance_path = "09 BPP.txt"
n_item, capacity, original_items = read_instance(instance_path)
print("n_item =", n_item)
print("capacity =", capacity)

n_item = 10
capacity = 10


## Next Fit Decreasing
Nel Next Fit Decreasing ordiniamo gli item in ordine decrescente di peso e manteniamo aperto un solo bin corrente.

Per ogni item:
1. se entra nel bin corrente, lo inseriamo;
2. altrimenti apriamo il bin successivo e lo inseriamo li.

E una euristica veloce, ma puo usare piu bin rispetto a First Fit.

In [2]:
def next_fit_decreasing(items, capacity):
    list_of_item = clone_items(items)
    residual_capacity = [capacity for _ in list_of_item]

    list_of_item.sort(key=lambda x: x.weight, reverse=True)

    print("start next fit insertion heuristic")
    next_bin = 0

    for item_i in list_of_item:
        if item_i.weight > residual_capacity[next_bin]:
            next_bin = next_bin + 1

        item_i.bin = next_bin
        residual_capacity[next_bin] = residual_capacity[next_bin] - item_i.weight
        print(item_i.index, " ", item_i.weight, " ", item_i.bin)

    tot_bin = 0
    for bin_res_cap in residual_capacity:
        if bin_res_cap < capacity:
            tot_bin = tot_bin + 1

    print("total bin used =", tot_bin)
    return list_of_item, tot_bin


next_fit_items, next_fit_tot_bin = next_fit_decreasing(original_items, capacity)

start next fit insertion heuristic
6   8   0
3   7   1
7   6   2
1   5   3
9   5   3
2   4   4
5   3   4
0   2   4
8   2   5
4   1   5
total bin used = 6


In [3]:
print("assegnazione next fit in ordine originale")
next_fit_by_index = sorted(next_fit_items, key=lambda x: x.index)
for item_i in next_fit_by_index:
    print("indice:", item_i.index, "peso:", item_i.weight, "bin:", item_i.bin)

assegnazione next fit in ordine originale
indice: 0 peso: 2 bin: 4
indice: 1 peso: 5 bin: 3
indice: 2 peso: 4 bin: 4
indice: 3 peso: 7 bin: 1
indice: 4 peso: 1 bin: 5
indice: 5 peso: 3 bin: 4
indice: 6 peso: 8 bin: 0
indice: 7 peso: 6 bin: 2
indice: 8 peso: 2 bin: 5
indice: 9 peso: 5 bin: 3


## First Fit Decreasing
Nel First Fit Decreasing ordiniamo gli item in ordine decrescente di peso e, per ogni item, cerchiamo il primo bin gia aperto in cui puo entrare.

Se non esiste nessun bin compatibile, apriamo un nuovo bin.

In generale ottiene una soluzione uguale o migliore del Next Fit, a costo di una ricerca piu ampia sui bin aperti.

In [4]:
def first_fit_decreasing(items, capacity):
    list_of_item = clone_items(items)
    residual_capacity = [capacity for _ in list_of_item]

    list_of_item.sort(key=lambda x: x.weight, reverse=True)

    print("start first fit insertion heuristic")

    for item_i in list_of_item:
        first_bin = 0
        while residual_capacity[first_bin] < item_i.weight:
            first_bin = first_bin + 1

        item_i.bin = first_bin
        residual_capacity[first_bin] = residual_capacity[first_bin] - item_i.weight
        print(item_i.index, " ", item_i.weight, " ", item_i.bin)

    tot_bin = 0
    for bin_res_cap in residual_capacity:
        if bin_res_cap < capacity:
            tot_bin = tot_bin + 1

    print("total bin used =", tot_bin)
    return list_of_item, tot_bin


first_fit_items, first_fit_tot_bin = first_fit_decreasing(original_items, capacity)

print("assegnazione first fit in ordine originale")
first_fit_by_index = sorted(first_fit_items, key=lambda x: x.index)
for item_i in first_fit_by_index:
    print("indice:", item_i.index, "peso:", item_i.weight, "bin:", item_i.bin)

print("confronto finale")
print("next fit total bin used =", next_fit_tot_bin)
print("first fit total bin used =", first_fit_tot_bin)

start first fit insertion heuristic
6   8   0
3   7   1
7   6   2
1   5   3
9   5   3
2   4   2
5   3   1
0   2   0
8   2   4
4   1   4
total bin used = 5
assegnazione first fit in ordine originale
indice: 0 peso: 2 bin: 0
indice: 1 peso: 5 bin: 3
indice: 2 peso: 4 bin: 2
indice: 3 peso: 7 bin: 1
indice: 4 peso: 1 bin: 4
indice: 5 peso: 3 bin: 1
indice: 6 peso: 8 bin: 0
indice: 7 peso: 6 bin: 2
indice: 8 peso: 2 bin: 4
indice: 9 peso: 5 bin: 3
confronto finale
next fit total bin used = 6
first fit total bin used = 5
